In [29]:
tokenized_data_folder_path = '/lustre/scratch126/cellgen/lotfollahi/DATASETS/gold/cell-graph-tokenizer/hst_corpus_90m/hst_corpus_90m_32_None_None_None_None_None_shifted_log_knn_10.dataset'

In [30]:
dataset = load_from_disk(tokenized_data_folder_path)

In [31]:
dataset

Dataset({
    features: ['rel_x_coord', 'rel_y_coord', 'cell_id', 'batch_token', 'gene_panel_token', 'assay_token', 'species_token', 'tissue_token', 'batch_value', 'gene_panel_value', 'assay_value', 'species_value', 'tissue_value', 'gene_tokens', 'gene_expr', 'seg_tokens', 'n_nonzero_tokens'],
    num_rows: 88626282
})

In [32]:
with open('/lustre/scratch126/cellgen/lotfollahi/DATASETS/tokenizer/hst_corpus_90m_32_None_None_None_None_None_shifted_log_knn_10_precomputed_split_train.pkl', 'rb') as f:
    train_idx = pickle.load(f)
dataset = dataset.select(train_idx)

In [33]:
dataset

Dataset({
    features: ['rel_x_coord', 'rel_y_coord', 'cell_id', 'batch_token', 'gene_panel_token', 'assay_token', 'species_token', 'tissue_token', 'batch_value', 'gene_panel_value', 'assay_value', 'species_value', 'tissue_value', 'gene_tokens', 'gene_expr', 'seg_tokens', 'n_nonzero_tokens'],
    num_rows: 81361647
})

In [ ]:
cell_ids = list(dataset['cell_id'])

In [21]:
import pickle

# Replace 'filename.pkl' with your file path
with open('/lustre/scratch126/cellgen/lotfollahi/DATASETS/tokenizer/hst_corpus_90m_32_None_None_None_None_None_shifted_log_knn_10_ordered_precomputed_split_train.pkl', 'rb') as f:
    data = pickle.load(f)

In [24]:
# Replace 'filename.pkl' with your file path
with open('/lustre/scratch126/cellgen/lotfollahi/DATASETS/tokenizer/hst_corpus_90m_32_None_None_None_None_None_shifted_log_knn_10_ordered_precomputed_split_test.pkl', 'rb') as f:
    data_test = pickle.load(f)

FileNotFoundError: [Errno 2] No such file or directory: '/lustre/scratch126/cellgen/lotfollahi/DATASETS/tokenizer/hst_corpus_90m_32_None_None_None_None_None_shifted_log_knn_10_ordered_precomputed_split_test.pkl'

In [23]:
data[-1]

81361646

Script exists for dataset-based cell sampling (stratified tissue sampling).

In [1]:
import os
import pickle
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
from datasets import concatenate_datasets, load_from_disk

/nfs/users/nfs_s/sb75/.venvs/nichejepa/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
plt.rcParams['figure.figsize'] = (12, 10)
plt.rcParams['figure.dpi'] = 300

In [3]:
corpus_name = 'hst_corpus_90m' # 'hst_corpus_80m'
tokenized_data_str = f'{corpus_name}_32_None_None_None_None_None_shifted_log_knn_10' # 'hst_corpus_80m_32_gene_corrected_read_depth_nonzero_mean_None_None_None_shifted_log_knn_10', 'hst_corpus_80m_32_None_None_None_None_None_shifted_log_knn_10'
tokenized_data_folder_path = f'/lustre/scratch126/cellgen/lotfollahi/DATASETS/gold/cell-graph-tokenizer/{corpus_name}/{tokenized_data_str}.dataset' # f'/nfs/team361/sb75/DATASETS/gold/cell-graph-tokenizer/hst_corpus_80m/{tokenized_data_str}.dataset'
silver_folder_path = '/nfs/team361/sb75/DATASETS/silver/'
tokenizer_folder_path = '/nfs/team361/sb75/DATASETS/tokenizer'

In [4]:
# Load dataset
dataset = load_from_disk(tokenized_data_folder_path)
dataset

Dataset({
    features: ['rel_x_coord', 'rel_y_coord', 'cell_id', 'batch_token', 'gene_panel_token', 'assay_token', 'species_token', 'tissue_token', 'batch_value', 'gene_panel_value', 'assay_value', 'species_value', 'tissue_value', 'gene_tokens', 'gene_expr', 'seg_tokens', 'n_nonzero_tokens'],
    num_rows: 88626282
})

In [5]:
dataset

Dataset({
    features: ['rel_x_coord', 'rel_y_coord', 'cell_id', 'batch_token', 'gene_panel_token', 'assay_token', 'species_token', 'tissue_token', 'batch_value', 'gene_panel_value', 'assay_value', 'species_value', 'tissue_value', 'gene_tokens', 'gene_expr', 'seg_tokens', 'n_nonzero_tokens'],
    num_rows: 88626282
})

## Dataset-based Cell Sampling

In [10]:
# Retrieve dataset IDs
dataset_ids = [cell_id.split('_')[0] for cell_id in list(dataset['cell_id'])]

In [11]:
# Read corpus statistics
data_artifacts_folder_path = '../../artifacts/data'
cohort_name = 'hst_corpus_90m'
dataset_stats_df = pd.read_csv(f'{data_artifacts_folder_path}/{cohort_name}/{cohort_name}_statistics.csv')
display(dataset_stats_df.head())

,cohort_name,dataset_name,dataset_id,batch,n_cells,n_genes,min_counts_per_cell,max_counts_per_cell,median_counts_per_cell,min_genes_per_cell,max_genes_per_cell,median_genes_per_cell,assay,species,tissue,condition,adata_batch_path
0,hst_corpus_90m,mhb14-1b_1p,14,batch0,689546,498,10.0,6004.0,342.0,10,366,123.0,merfish,homo_sapiens,breast,invasive_ductal_carcinoma,/nfs/team361/sb75/DATASETS/silver/mhb14-1b_1p/...
1,hst_corpus_90m,mhc15-2b_1p,15,batch0,649711,498,10.0,5320.0,352.0,10,335,118.0,merfish,homo_sapiens,colon,adenocarcinoma,/nfs/team361/sb75/DATASETS/silver/mhc15-2b_1p/...
2,hst_corpus_90m,mhc15-2b_1p,15,batch1,779379,498,10.0,10207.0,329.0,10,401,113.0,merfish,homo_sapiens,colon,adenocarcinoma,/nfs/team361/sb75/DATASETS/silver/mhc15-2b_1p/...
3,hst_corpus_90m,mhl16-2b_1p,16,batch0,582921,498,10.0,3483.0,289.0,10,292,92.0,merfish,homo_sapiens,liver,hepatocellular_carcinoma,/nfs/team361/sb75/DATASETS/silver/mhl16-2b_1p/...
4,hst_corpus_90m,mhl16-2b_1p,16,batch1,531649,498,10.0,5241.0,259.0,10,330,93.0,merfish,homo_sapiens,liver,hepatocellular_carcinoma,/nfs/team361/sb75/DATASETS/silver/mhl16-2b_1p/...


In [12]:
random.seed(0)
dataset_ids_set = set(dataset_ids)
sampled_dataset_ids_set = random.sample(dataset_ids_set, len(dataset_ids_set))
#sampled_dataset_ids_set

/tmp/ipykernel_2404835/628866144.py:3: DeprecationWarning: Sampling from a set deprecated
since Python 3.9 and will be removed in a subsequent version.
  sampled_dataset_ids_set = random.sample(dataset_ids_set, len(dataset_ids_set))


In [13]:
num_workers = 12
sampled_dataset_ids = []
datasets_list = []

for sample_ratio in [0.01, 0.1, 0.25, 0.5]:
    print(f"Sample ratio: {sample_ratio}.")
    tissue_cell_count_dict = {}
    tissue_cell_count_diff_dict = {}
    max_tissue_cell_count = (dataset_stats_df.groupby('tissue')['n_cells'].sum() * sample_ratio).astype(int)
    for dataset_id in sampled_dataset_ids_set:
        dataset_tissue = dataset_stats_df[dataset_stats_df['dataset_id'] == int(dataset_id)]['tissue'].values[0]
        dataset_cell_count = dataset_stats_df[dataset_stats_df['dataset_id'] == int(dataset_id)]['n_cells'].values.sum()
        if dataset_tissue not in tissue_cell_count_dict:
            tissue_cell_count_dict[dataset_tissue] = 0
            tissue_cell_count_diff_dict[dataset_tissue] = max_tissue_cell_count[dataset_tissue]
        if tissue_cell_count_diff_dict[dataset_tissue] >= 0:
            current_dataset = dataset.filter(
                lambda ids: [i.split('_')[0] == dataset_id for i in ids],
                input_columns="cell_id",
                batched=True,
                batch_size=10000,
                num_proc=num_workers,
                desc="Filtering by dataset_id"
            )
            print(f"Complete length of current dataset: {len(current_dataset)}.")
            tissue_cell_count_dict[dataset_tissue] += dataset_cell_count
            tissue_cell_count_diff_dict[dataset_tissue] = max_tissue_cell_count[dataset_tissue] - tissue_cell_count_dict[dataset_tissue]
            if tissue_cell_count_diff_dict[dataset_tissue] < 0:
                current_dataset = current_dataset.select(range(0, len(current_dataset) + tissue_cell_count_diff_dict[dataset_tissue]))
                print(f"Tissue cell count diff: {tissue_cell_count_diff_dict[dataset_tissue]}.")    
                print(f"Truncated length of current dataset: {len(current_dataset)}.")
            datasets_list.append(current_dataset)
        else:
            print(f"Tissue cell count diff: {tissue_cell_count_diff_dict[dataset_tissue]}. Continuing...") 
            continue
            
    combined_dataset = concatenate_datasets(datasets_list)
    print(f"Combined dataset length: {len(combined_dataset)}.")                                            
    print('Saving combined dataset...')
    break
    #combined_dataset.save_to_disk(tokenized_data_folder_path.replace('hst_corpus_90m_32_None_None_None_None_None_shifted_log_knn_10', f'hst_corpus_90m_{sample_ratio}_sample_32_None_None_None_None_None_shifted_log_knn_10'))

Sample ratio: 0.01.
Complete length of current dataset: 644736.
Tissue cell count diff: -611785.
Truncated length of current dataset: 32951.
Complete length of current dataset: 777955.
Tissue cell count diff: -770176.
Truncated length of current dataset: 7779.
Complete length of current dataset: 184400.
Tissue cell count diff: -166027.
Truncated length of current dataset: 18373.
Complete length of current dataset: 1434182.
Tissue cell count diff: -1293740.
Truncated length of current dataset: 140442.
Complete length of current dataset: 1156748.
Tissue cell count diff: -1055255.
Truncated length of current dataset: 101493.
Tissue cell count diff: -611785. Continuing...
Complete length of current dataset: 1139261.
Tissue cell count diff: -1096297.
Truncated length of current dataset: 42964.
Tissue cell count diff: -1055255. Continuing...
Complete length of current dataset: 272074.
Tissue cell count diff: -220547.
Truncated length of current dataset: 51527.
Tissue cell count diff: -220547

In [16]:
combined_dataset

Dataset({
    features: ['rel_x_coord', 'rel_y_coord', 'cell_id', 'batch_token', 'gene_panel_token', 'assay_token', 'species_token', 'tissue_token', 'batch_value', 'gene_panel_value', 'assay_value', 'species_value', 'tissue_value', 'gene_tokens', 'gene_expr', 'seg_tokens', 'n_nonzero_tokens'],
    num_rows: 886253
})

In [17]:
combined_dataset['n_nonzero_tokens']

Column([800, 716, 716, 1183, 1359])

In [ ]:
sampled_dataset = dataset.filter(
    lambda ids: [i.split('_')[0] in dataset_ids_sample_set for i in ids],
    input_columns="cell_id",
    batched=True,
    batch_size=10000,
    num_proc=num_workers,
    desc="Filtering by dataset_id"
)

In [ ]:
sampled_dataset

In [ ]:
num_workers = 12

def keep_row(x):
    return torch.isin(torch.tensor(int(x["cell_id"][0].split('_')[0])), dataset_ids_tensor)

sampled_dataset = dataset.filter(
    keep_row,
    batched=True,
    batch_size=10000, 
    num_proc=num_workers)
print(sampled_dataset)

## Stratified Cell Sampling

In [ ]:
# Define sample config
n_samples = 30
sample_type = 'frac_per_bin' # 'cells_per_bin', 'frac_per_bin', 'mixed'
avg_cells_per_bin = 50
sampled_cells_per_bin = 1
sample_frac = 0.1

In [ ]:
# Retrieve cell IDs
dataset_cell_ids = dataset['cell_id']

In [ ]:
# Create cell ID dict
cell_id_dict = {v: i for i, v in enumerate(dataset_cell_ids)} # for speedup

In [ ]:
all_epochs_indices = [[] for i in range(n_samples)]

for subdir, _, files in os.walk(silver_folder_path + corpus_name):
    for file_idx, file in enumerate(files):
        if file.endswith('.h5ad'):
            file_path = os.path.join(subdir, file)
            adata = sc.read_h5ad(file_path)
            print('====================================================================================================')
            print(f'Processing adata with file path {file_path}...')

            # Retrieve x and y bins
            print('Creating bins...')
            n_bins = int(np.round(len(adata) / avg_cells_per_bin, decimals=0))
            x_bins = np.linspace(adata.obsm['spatial'][:, 0].min(), adata.obsm['spatial'][:, 0].max(), int(np.round(np.sqrt(n_bins), decimals=0)) + 1)
            y_bins = np.linspace(adata.obsm['spatial'][:, 1].min(), adata.obsm['spatial'][:, 1].max(), int(np.round(np.sqrt(n_bins), decimals=0)) + 1)

            # Assign each point to a bin
            adata.obs['x_bin'] = pd.cut(adata.obsm['spatial'][:, 0], bins=x_bins, labels=False, include_lowest=True)
            adata.obs['y_bin'] = pd.cut(adata.obsm['spatial'][:, 1], bins=y_bins, labels=False, include_lowest=True)

            print('Sampling cells...')
            indices = [] 
            for i in range(n_samples):
                if sample_type == 'cells_per_bin':
                    sampled_df = adata.obs.groupby(['x_bin', 'y_bin']).apply(lambda group: group.sample(n=sampled_cells_per_bin))
                elif sample_type == 'frac_per_bin':
                    sampled_df = adata.obs.groupby(['x_bin', 'y_bin']).apply(lambda group: group.sample(frac=sample_frac))
                elif sample_type == 'mixed':
                    # First, sample 1 point from each group
                    min_sampled_df = adata.obs.groupby(['x_bin', 'y_bin']).apply(lambda group: group.sample(n=sampled_cells_per_bin))
                    
                    # Then, sample the remaining points based on sample_frac
                    remaining_df = adata.obs.groupby(['x_bin', 'y_bin']).apply(lambda group: group.sample(frac=sample_frac))
                    
                    # Combine both sampled DataFrames
                    sampled_df = pd.concat([min_sampled_df, remaining_df]).drop_duplicates()
                
                sampled_df = sampled_df.reset_index(drop=True)
            
                #adata.obs['sampled_flag'] = adata.obs['cell_id'].apply(lambda x: x in sampled_df['cell_id'].values.tolist())
    
                indices.append([cell_id_dict.get(item, None) for item in sampled_df['cell_id'].values.tolist()])
                del(sampled_df)
            
                #sc.pl.spatial(adata,
                #              color='sampled_flag',
                #              spot_size=spot_size)

            del(adata)
            for i, epoch_indices in enumerate(indices):
                all_epochs_indices[i].extend(epoch_indices)

            print('Saving results to disk...')
            with open(f'{tokenizer_folder_path}/{tokenized_data_str}_precomputed_epoch_split_{n_samples}.pkl', 'wb') as f:
                pickle.dump(all_epochs_indices, f)

all_epochs_n_nonzero_tokens = []

for epoch_indices in all_epochs_indices:
    epoch_data = dataset.select(epoch_indices)
    epoch_n_nonzero_tokens = epoch_data['n_nonzero_tokens']
    all_epochs_n_nonzero_tokens.append(epoch_n_nonzero_tokens)

with open(f'{tokenizer_folder_path}/{tokenized_data_str}_precomputed_epoch_n_nonzero_tokens_{n_samples}.pkl', 'wb') as f:
    pickle.dump(all_epochs_n_nonzero_tokens, f)

In [ ]:
# Sample batches
for file_path_sample_ratio in [0.01, 0.1, 0.25, 0.5]:
    n_samples = int(30 / file_path_sample_ratio)
    file_paths = []
    
    # for subdir, _, files in os.walk(silver_folder_path + 'hst_corpus_70m'):
    for subdir, _, files in os.walk(silver_folder_path + 'hst_corpus_80m'):
        for file_idx, file in enumerate(files):
            if file.endswith('.h5ad'):
                file_path = os.path.join(subdir, file)
                file_paths.append(file_path)
    
    file_path_sample_n = max(1, int(len(file_paths) * file_path_sample_ratio))
    sampled_file_paths = random.sample(file_paths, file_path_sample_n)
    
    all_epochs_indices = [[] for i in range(n_samples)]
    
    # for subdir, _, files in os.walk(silver_folder_path + 'hst_corpus_70m'):
    for subdir, _, files in os.walk(silver_folder_path + 'hst_corpus_80m'):
        for file_idx, file in enumerate(files):
            if file.endswith('.h5ad'):
                file_path = os.path.join(subdir, file)
                if file_path in sampled_file_paths:
                    adata = sc.read_h5ad(file_path)
                    print('====================================================================================================')
                    print(f'Processing adata with file path {file_path}...')
        
                    # Retrieve x and y bins
                    print('Creating bins...')
                    n_bins = int(np.round(len(adata) / avg_cells_per_bin, decimals=0))
                    x_bins = np.linspace(adata.obsm['spatial'][:, 0].min(), adata.obsm['spatial'][:, 0].max(), int(np.round(np.sqrt(n_bins), decimals=0)) + 1)
                    y_bins = np.linspace(adata.obsm['spatial'][:, 1].min(), adata.obsm['spatial'][:, 1].max(), int(np.round(np.sqrt(n_bins), decimals=0)) + 1)
        
                    # Assign each point to a bin
                    adata.obs['x_bin'] = pd.cut(adata.obsm['spatial'][:, 0], bins=x_bins, labels=False, include_lowest=True)
                    adata.obs['y_bin'] = pd.cut(adata.obsm['spatial'][:, 1], bins=y_bins, labels=False, include_lowest=True)
        
                    print('Sampling cells...')
                    indices = [] 
                    for i in range(n_samples):
                        if sample_type == 'cells_per_bin':
                            sampled_df = adata.obs.groupby(['x_bin', 'y_bin']).apply(lambda group: group.sample(n=sampled_cells_per_bin))
                        elif sample_type == 'frac_per_bin':
                            sampled_df = adata.obs.groupby(['x_bin', 'y_bin']).apply(lambda group: group.sample(frac=sample_frac))
                        elif sample_type == 'mixed':
                            # First, sample 1 point from each group
                            min_sampled_df = adata.obs.groupby(['x_bin', 'y_bin']).apply(lambda group: group.sample(n=sampled_cells_per_bin))
                            
                            # Then, sample the remaining points based on sample_frac
                            remaining_df = adata.obs.groupby(['x_bin', 'y_bin']).apply(lambda group: group.sample(frac=sample_frac))
                            
                            # Combine both sampled DataFrames
                            sampled_df = pd.concat([min_sampled_df, remaining_df]).drop_duplicates()
                        
                        sampled_df = sampled_df.reset_index(drop=True)
                    
                        #adata.obs['sampled_flag'] = adata.obs['cell_id'].apply(lambda x: x in sampled_df['cell_id'].values.tolist())
            
                        indices.append([cell_id_dict.get(item, None) for item in sampled_df['cell_id'].values.tolist()])
                        del(sampled_df)
                    
                        #sc.pl.spatial(adata,
                        #              color='sampled_flag',
                        #              spot_size=spot_size)
        
                    del(adata)
                    for i, epoch_indices in enumerate(indices):
                        all_epochs_indices[i].extend(epoch_indices)
        
                    print('Saving results to disk...')
                    with open(f'{tokenizer_folder_path}/{tokenized_data_str}_precomputed_epoch_split_{int(file_path_sample_ratio * 100)}_pct_subsample_{n_samples}.pkl', 'wb') as f:
                        pickle.dump(all_epochs_indices, f)
        
    all_epochs_n_nonzero_tokens = []
    
    for epoch_indices in all_epochs_indices:
        epoch_data = dataset.select(epoch_indices)
        epoch_n_nonzero_tokens = epoch_data['n_nonzero_tokens']
        all_epochs_n_nonzero_tokens.append(epoch_n_nonzero_tokens)
    
    with open(f'{tokenizer_folder_path}/{tokenized_data_str}_precomputed_epoch_n_nonzero_tokens_{int(file_path_sample_ratio * 100)}_pct_subsample_{n_samples}.pkl', 'wb') as f:
        pickle.dump(all_epochs_n_nonzero_tokens, f)

In [ ]:
file_path_sample_ratio = 0.25
n_samples = int(1 / file_path_sample_ratio) * 10

with open(f'/lustre/scratch126/cellgen/team361/DATASETS/tokenizer/hst_corpus_70m_32_None_None_None_None_None_shifted_log_knn_10_precomputed_epoch_split_{int(file_path_sample_ratio * 100)}_pct_subsample_{n_samples}.pkl', 'rb') as file:
    precomputed_epoch_split_subsample_full = pickle.load(file)

with open(f'/lustre/scratch126/cellgen/team361/DATASETS/tokenizer/hst_corpus_70m_32_None_None_None_None_None_shifted_log_knn_10_precomputed_epoch_n_nonzero_tokens_{int(file_path_sample_ratio * 100)}_pct_subsample_{n_samples}.pkl', 'rb') as file:
    precomputed_epoch_n_nonzero_tokens_subsample_full = pickle.load(file)

with open(f'/lustre/scratch126/cellgen/team361/DATASETS/tokenizer/hst_corpus_70m_32_None_None_None_None_None_shifted_log_knn_10_precomputed_epoch_split_{int(file_path_sample_ratio * 100)}_pct_subsample_30.pkl', 'rb') as file:
    precomputed_epoch_split_subsample_30 = pickle.load(file)

with open(f'/lustre/scratch126/cellgen/team361/DATASETS/tokenizer/hst_corpus_70m_32_None_None_None_None_None_shifted_log_knn_10_precomputed_epoch_n_nonzero_tokens_{int(file_path_sample_ratio * 100)}_pct_subsample_30.pkl', 'rb') as file:
    precomputed_epoch_n_nonzero_tokens_subsample_30 = pickle.load(file)

In [ ]:
len(precomputed_epoch_split_subsample_full)

In [ ]:
len(precomputed_epoch_n_nonzero_tokens_subsample_full)

In [ ]:
len(precomputed_epoch_split_subsample_full[0])

In [ ]:
len(precomputed_epoch_n_nonzero_tokens_subsample_full[0])

In [ ]:
len(precomputed_epoch_split_subsample_full[-1])

In [ ]:
len(precomputed_epoch_n_nonzero_tokens_subsample_full[-1])

In [ ]:
len(precomputed_epoch_split_subsample_30[0])

In [ ]:
len(precomputed_epoch_n_nonzero_tokens_subsample_30[0])

In [ ]:
precomputed_epoch_split_subsample_full_new = precomputed_epoch_split_subsample_30 + precomputed_epoch_split_subsample_full[30:]
precomputed_epoch_n_nonzero_tokens_subsample_full_new = precomputed_epoch_n_nonzero_tokens_subsample_30 + precomputed_epoch_n_nonzero_tokens_subsample_full[30:]

In [ ]:
with open(f'/lustre/scratch126/cellgen/team361/DATASETS/tokenizer/{tokenized_data_str}_precomputed_epoch_split_{int(file_path_sample_ratio * 100)}_pct_subsample_{n_samples}.pkl', 'wb') as f:
    pickle.dump(precomputed_epoch_split_subsample_full_new, f)

with open(f'/lustre/scratch126/cellgen/team361/DATASETS/tokenizer/{tokenized_data_str}_precomputed_epoch_n_nonzero_tokens_{int(file_path_sample_ratio * 100)}_pct_subsample_{n_samples}.pkl', 'wb') as f:
    pickle.dump(precomputed_epoch_n_nonzero_tokens_subsample_full_new, f)

In [ ]:
with open(f'/lustre/scratch126/cellgen/team361/DATASETS/tokenizer/{tokenized_data_str}_precomputed_epoch_split_{int(file_path_sample_ratio * 100)}_pct_subsample_{n_samples}.pkl', 'rb') as file:
    precomputed_epoch_split_subsample_full_new = pickle.load(file)

with open(f'/lustre/scratch126/cellgen/team361/DATASETS/tokenizer/{tokenized_data_str}_precomputed_epoch_n_nonzero_tokens_{int(file_path_sample_ratio * 100)}_pct_subsample_{n_samples}.pkl', 'rb') as file:
    precomputed_epoch_n_nonzero_tokens_subsample_full_new = pickle.load(file)

In [ ]:
len(precomputed_epoch_split_subsample_full_new)

In [ ]:
len(precomputed_epoch_n_nonzero_tokens_subsample_full_new)

In [ ]:
len(precomputed_epoch_split_subsample_full_new[-1])

In [ ]:
len(precomputed_epoch_n_nonzero_tokens_subsample_full_new[-1])